<a href="https://colab.research.google.com/github/aishwaryasuriyakumar/aishwaryasrcs09-codeboosters-internship-2026/blob/main/DAY_05_mini_project_students__1_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
df=pd.read_csv('student_performance.csv')

#printing first 5 rows
df.head()

df.isnull().any()

df.isnull().sum()

print("Numbr of  Rows  and columns:",df.shape)
print("NUmber of Rows:", df.shape[0])
print("Number of Columns:",df.shape[1])

print(df.columns.tolist())

df.columns.tolist()

df.describe()

df.dtypes



#Transform the Student performance data


In [ ]:
"""#Transform the Student performance data"""
df=df.dropna()

df=df.drop_duplicates()

df['total_score']=df['english_score']+df[ 'math_score']+df['programming_score']+df[ 'science_score']

df['avg_score']=df['total_score']/4
print("Total Score:\n",df[['name','total_score']])
print("Average score:\n",df[['name','avg_score']])

import sqlite3
print("Database created and successfully connected to SQLite")
conn=sqlite3.connect('student_details.db')

df.to_sql('student_details',conn,if_exists='replace',index=False)



#Five Analytical queries


In [ ]:
"""#Five Analytical queries"""
#Top 5 students with respect to total mark
query1="""select name,total_score from student_details order by total_score limit 5"""
result1=pd.read_sql(query1,conn)
print("Top 5 students with respect to total mark:\n",result1)

#Department Average
query2="""select department,avg(total_score)as dept_avg,count(*) as total_students from student_details
group by department order by dept_avg desc"""
result2=pd.read_sql(query2,conn)
print("Departmnt AVerage:\n",result2)

query3="""select name,attendance_percentage from student_details where attendance_percentage>90"""
result3=pd.read_sql(query3,conn)
print("Students with attendance percentage greater than 90:\n",result3)

query4="""select name,department,programming_score from student_details order by programming_score desc limit 5"""
result4=pd.read_sql(query4,conn)
print("Top 5 students with respect to programming score:\n",result4)

#student count by department
query5="""select department,count(*) as students_count from student_details group by department
order by students_count desc"""
result5=pd.read_sql_query(query5,conn)
print("Department wise students count:\n",result5)



#Task 3:Bar charts
1.Department Average
2.Attendance_percentage by department
3. Total marks by gender


In [ ]:

import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(18,12))
fig.suptitle("STUDENT  PERFORMANCE  DASHBOARD",fontsize=20,fontweight='bold',color='black')
ax1= axes[0]
colors4=['#c27e79','#8E3FE2','#00adb5','#DFB722']
ax1.set_ylim(0,400)
bars1= ax1.bar(result2['department'],result2['dept_avg'],color=colors4[:len('department')],edgecolor='black')
for bars in bars1:
   ax1.text(bars.get_x()+bars.get_width()/2,bars.get_height()*1.01,f'{bars.get_height():.2f}',ha='center',fontweight='bold',fontsize=12,color='#DFB722')
   ax1.set_title("Department Average marks",fontsize=14,fontweight='bold')
   ax1.set_ylabel("Department Average Marks")
   ax1.set_xlabel("Departments")

ax2=axes[1]
ax2.set_title("Student Count by Department",fontsize=14,fontweight='bold')
ax2.pie(
    result5['students_count'],
    labels=result5['department'],
    autopct=lambda p: f'{int(round(p*sum(result5["students_count"])/100))}',
    colors=colors4[:len(result5)],
    startangle=90
)
plt.show()



#4.ML Prediction-Train RandomForest and predict Programming score


In [ ]:
"""#4.ML Prediction-Train RandomForest and predict Programming score"""
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score,mean_squared_error
from  sklearn.model_selection import train_test_split

rf_model=RandomForestRegressor(n_estimators=100,random_state=42)

df=pd.read_csv('student_performance.csv')
x=df[['english_score','science_score','math_score']]
y=df['programming_score']
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)
rf_model.fit(x_train,y_train)
predictions=rf_model.predict(x_test)
print("MAE:", mean_absolute_error(y_test,predictions ))
print("MSE:",mean_squared_error(y_test,predictions))
print("R² Score:", r2_score(y_test, predictions))



#5.Load the original data into another file



In [ ]:

df.to_csv('Student_performance_details',index=False)
print("Cleaned data saved to :Student_performance_details.csv")



#Medallion Architecture
1.Bronze

2.Silver

3.Gold


In [ ]:

!pip install pyspark -quiet

from pyspark .sql import SparkSession
from pyspark.sql  import functions
import pandas as pd
from pyspark.sql.functions import year, month,to_date,col,round as spark_round,avg
import warnings
warnings.filterwarnings('ignore')

spark=SparkSession.builder.appName('Day5_Mini_Project')\
.config('spark.sql.adaptive.enabled','true').getOrCreate()

print("Spark session:ACTIVE")
print("Spark versoin:",spark.version)
print("Application Name:",spark.sparkContext.appName)

df_bronze=spark.read.option('header','true')\
.option('inferSchema','true')\
.csv('student_performance.csv')

print("===Bronze Medallion===")
df_bronze.show(5)
print("Number of Rows:",df_bronze.count())
print("Number of Columns:",(len(df_bronze.columns)))
df_bronze.printSchema()

df_bronze.write.mode('overwrite').parquet('bronze_students.parquet')
print("Bronze file saved as bronze_students.parquet")

df_silver=df_bronze.drop_duplicates().dropna()
df_silver=df_bronze.withColumn('total_score',(col('english_score')+
col('science_score')+col('math_score')+col('programming_score')))
df_silver=df_silver.withColumn('avg_score',spark_round(col('total_score')/4,2))
print("New Columns added:Total_score,Avg_score")
df_silver.select('total_score', 'avg_score').show(5)

df_silver.write.mode('overwrite').parquet('students_silver.parquet')
print("Silver parquet saved successfully")

#Top 5 students by average score
print("Top 5 Average marks students")
top_avg=df_silver.select('name','avg_score')\
.orderBy(col('avg_score'),ascending=False)
top_avg.show(5)

att_per=df_silver.groupBy('gender')\
.agg(avg('attendance_percentage').alias('avg_attendance'))\
.orderBy('avg_attendance',ascending=False)
att_per.show(5)

gold_region=top_avg
gold_region.write.mode('overwrite').parquet('gold_students.parquet')
print("Gold 1 saved:gold_students.parquet saved successfully")

gold_attendance=att_per
gold_attendance.write.mode('overwrite').parquet('gold_attendance_students.parquet')
print("Gold 2 saved:gold_attendance_students.parquet saved successfully")
